In [ ]:
# clean songs and split into 10 chunks
import pandas as pd
import os
import tempfile
import math

# -------- SETTINGS --------
INPUT_FILE = "/Users/meganvaughn/Downloads/archive/songs.csv"        # original huge file
N_CHUNKS = 10                   # EXACT number of output chunks
READ_CHUNK = 200_000            # read size (memory-safe)
MIN_LYRICS_LEN = 50
MAX_LYRICS_LEN = 1000
RANDOM_STATE = 42

# writable output directory
OUT_DIR = os.path.join(tempfile.gettempdir(), "clean_chunks")
os.makedirs(OUT_DIR, exist_ok=True)
print("Writing chunks to:", OUT_DIR)

# columns to keep
KEEP_COLS = [
    "track_id", "track_name",
    "artist_id", "artist_name",
    "genre", "lyrics",
    "danceability", "energy", "valence",
    "speechiness", "acousticness",
    "instrumentalness", "liveness",
    "loudness", "tempo", "duration_ms",
    "key", "mode"
]

cleaned_parts = []

# -------- PASS 1: STREAM + CLEAN --------
for chunk in pd.read_csv(INPUT_FILE, chunksize=READ_CHUNK):

    # keep only needed columns
    keep = [c for c in KEEP_COLS if c in chunk.columns]
    chunk = chunk[keep]

    # drop missing genre/lyrics
    chunk = chunk.dropna(subset=["genre", "lyrics"])

    # remove short lyrics
    chunk = chunk[chunk["lyrics"].astype(str).str.len() >= MIN_LYRICS_LEN]

    # truncate lyrics
    chunk["lyrics"] = chunk["lyrics"].astype(str).str.slice(0, MAX_LYRICS_LEN)

    # numeric coercion
    for col in [
        "danceability","energy","valence","speechiness",
        "acousticness","instrumentalness","liveness",
        "loudness","tempo","duration_ms"
    ]:
        if col in chunk.columns:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    # enforce valid ranges
    if "danceability" in chunk.columns:
        chunk = chunk[chunk["danceability"].between(0, 1)]
    if "energy" in chunk.columns:
        chunk = chunk[chunk["energy"].between(0, 1)]
    if "valence" in chunk.columns:
        chunk = chunk[chunk["valence"].between(0, 1)]
    if "tempo" in chunk.columns:
        chunk = chunk[chunk["tempo"].between(40, 250)]
    if "loudness" in chunk.columns:
        chunk = chunk[chunk["loudness"].between(-60, 0)]
    if "duration_ms" in chunk.columns:
        chunk = chunk[chunk["duration_ms"].between(10_000, 1_200_000)]

    # deduplicate within chunk
    if "track_id" in chunk.columns:
        chunk = chunk.drop_duplicates("track_id")

    cleaned_parts.append(chunk)

# combine all cleaned data
cleaned = pd.concat(cleaned_parts, ignore_index=True)

# global deduplication
if "track_id" in cleaned.columns:
    cleaned = cleaned.drop_duplicates("track_id")

print("Total cleaned rows:", len(cleaned))

# shuffle for even distribution
cleaned = cleaned.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# -------- PASS 2: SPLIT INTO EXACTLY 10 FILES --------
rows_per_chunk = math.ceil(len(cleaned) / N_CHUNKS)

for i in range(N_CHUNKS):
    start = i * rows_per_chunk
    end = min((i + 1) * rows_per_chunk, len(cleaned))
    out = cleaned.iloc[start:end]

    out_path = os.path.join(OUT_DIR, f"chunk_{i+1}.csv")
    out.to_csv(out_path, index=False)

    print(f"Wrote {out_path} ({len(out)} rows)")

print("Done.")


Writing chunks to: /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks
Total cleaned rows: 549925
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_1.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_2.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_3.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_4.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_5.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_6.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_7.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_8.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/clean_chunks/chunk_9.csv (54993 rows)
Wrote /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/

In [4]:
# clean artists
import pandas as pd
import os
import tempfile

# ---- EDIT THIS ----
ARTISTS_FILE = "/Users/meganvaughn/Downloads/archive/artists.csv"   # path to your artists csv on your local machine

# writable output folder
OUT_DIR = os.path.join(tempfile.gettempdir(), "spotify_clean_for_viya")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_FILE = os.path.join(OUT_DIR, "artists_clean.csv")
print("Will write to:", OUT_FILE)

# read
df = pd.read_csv(ARTISTS_FILE)

# --- clean strings ---
df["id"] = df["id"].astype(str).str.strip()
df["name"] = df["name"].astype(str).str.strip()

if "genres" in df.columns:
    df["genres"] = df["genres"].astype(str).str.strip()

if "main_genre" in df.columns:
    df["main_genre"] = df["main_genre"].astype(str).str.strip()

# --- numeric coercion ---
df["followers"] = pd.to_numeric(df["followers"], errors="coerce")
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce")

# --- drop missing critical fields ---
df = df.dropna(subset=["id", "name", "popularity"])

# --- enforce valid ranges ---
df = df[(df["followers"] >= 0) & (df["popularity"].between(0, 100))]

# --- standardize main genre + fill missing ---
df["main_genre_clean"] = (
    df["main_genre"]
      .fillna("")
      .astype(str)
      .str.strip()
      .str.upper()
)

df.loc[df["main_genre_clean"].isin(["", "NONE", "NAN"]), "main_genre_clean"] = pd.NA
df["genre_final"] = df["main_genre_clean"].fillna("OTHER")

# --- dedupe artists by id (keep highest followers, then popularity) ---
df = df.sort_values(["followers", "popularity"], ascending=False)
df = df.drop_duplicates(subset=["id"], keep="first")

# write
df.to_csv(OUT_FILE, index=False)

print("Cleaned artists shape:", df.shape)
print(df.head(3))


Will write to: /var/folders/k3/50xysxys5c73rykz0k1vjjbc0000gn/T/spotify_clean_for_viya/artists_clean.csv
Cleaned artists shape: (71440, 8)
                           id          name  followers  popularity  \
14704  4YRxDV8wJFPHPTeXepOstw  Arijit Singh  168508682          92   
1605   06HL4z0CvFAxyc27GXpf02  Taylor Swift  148253383         100   
1575   6eUKZXaKkcviH0Ku9w2n3V    Ed Sheeran  123790754          90   

                                                 genres main_genre  \
14704  ['hindi pop', 'bollywood', 'desi', 'bangla pop']        Pop   
1605                                                 []        Pop   
1575                                       ['soft pop']        Pop   

      main_genre_clean genre_final  
14704              POP         POP  
1605               POP         POP  
1575               POP         POP  
